In [ ]:
import math
import random

# -------------------
# Cities Coordinates
# -------------------
cities = {
    0:(0,0), 1:(3,4), 2:(6,1),
    3:(7,5), 4:(2,7), 5:(5,3)
}

# -------------------
# Distance Functions
# -------------------
def distance(a, b):
    x1, y1 = a
    x2, y2 = b
    return math.sqrt((x1 - x2)**2 + (y1 - y2)**2)

def total_distance(route):
    size = len(route)
    dist = 0
    for i in range(size - 1):
        dist += distance(cities[route[i]], cities[route[i+1]])
    dist += distance(cities[route[-1]], cities[route[0]])  # return to start
    return dist

# -------------------
# Initial Population
# -------------------
def init_population(size):
    population = []
    nodes = list(cities.keys())
    for _ in range(size):
        ind = nodes[:]
        random.shuffle(ind)
        population.append(ind)
    return population

# -------------------
# Tournament Selection
# -------------------
def tournament_selection(pop, k=3):
    selected = random.sample(pop, k)
    return min(selected, key=total_distance)

# -------------------
# Cycle Crossover
# -------------------
def cycle_crossover(p1, p2):
    size = len(p1)
    c1, c2 = [None]*size, [None]*size

    # Helper to create one child
    def create_child(parent1, parent2):
        child = [None]*size
        cycles_done = [False]*size
        index = 0
        while not all(cycles_done):
            # start new cycle
            while cycles_done[index]:
                index += 1
            start = index
            val = parent1[start]
            while True:
                child[index] = parent1[index]
                cycles_done[index] = True
                val2 = parent2[index]
                index = parent1.index(val2)
                if index == start:
                    break
        # Fill remaining None positions from parent2
        for i in range(size):
            if child[i] is None:
                child[i] = parent2[i]
        return child

    c1 = create_child(p1, p2)
    c2 = create_child(p2, p1)
    return c1, c2

# -------------------
# Inversion Mutation
# -------------------
def inversion_mutation(ind, rate=0.1):
    if random.random() < rate:
        i, j = sorted(random.sample(range(len(ind)), 2))
        ind[i:j+1] = reversed(ind[i:j+1])
    return ind

# -------------------
# Genetic Algorithm
# -------------------
def genetic_algo(pop_size=10, generations=100):
    population = init_population(pop_size)
    best = min(population, key=total_distance)
    best_dist = total_distance(best)

    for gen in range(generations):
        new_pop = []

        while len(new_pop) < len(population):
            p1 = tournament_selection(population)
            p2 = tournament_selection(population)
            c1, c2 = cycle_crossover(p1, p2)

            c1 = inversion_mutation(c1, 0.1)
            c2 = inversion_mutation(c2, 0.1)
            new_pop.extend([c1, c2])

        population = new_pop[:pop_size]

        current_best = min(population, key=total_distance)
        curr_distance = total_distance(current_best)

        if curr_distance < best_dist:
            best_dist = curr_distance
            best = current_best

        print(f"Gen {gen+1}: Best Distance = {best_dist:.2f}")

    return best, best_dist

# -------------------
# Run GA
# -------------------
best_route, best_distance = genetic_algo(pop_size=10, generations=100)
print("\nBest Route:", best_route)
print("Best Distance:", best_distance)